# PC 5: Iterative methods for solving linear systems

**<big><font color=black><span style="background-color:skyblue">To hand in</span></font> after the PC: Exercise 3, Question 2</big>**

In [ ]:
# This package provides efficient array operations
import numpy as np
# This package provides linear algebra functions (inverse, determinant, ...)
import numpy.linalg as lin
# This package provides sparse-matrix tools
import scipy.sparse as sparse
# This function computes eigenvalues of a sparse matrix
from scipy.sparse.linalg import eigsh as sparse_eigs

# This package provides plotting tools
from matplotlib import pyplot as plt


>**Reminders and notation:**
>
>- Given a norm $\left\Vert \cdot \right\Vert$ on $\mathbb K^n$ ($\mathbb K = \mathbb R$ or $\mathbb C$), the associated subordinate norm on $M_n(\mathbb{K})$ is defined by
>
>$$\left\Vert A \right\Vert = \sup_{\substack{x\in\mathbb{K}^n \\ x\neq 0}} \frac{\left\Vert Ax \right\Vert}{\left\Vert x \right\Vert} = \max_{\substack{x\in\mathbb{K}^n \\ \left\Vert x \right\Vert= 1}}\left\Vert Ax \right\Vert.$$
>
>- A norm $\left\Vert \cdot \right\Vert$ on $M_n(\mathbb K)$ is called a matrix norm (or algebra norm) if
>
>$$\forall A,B \in M_n(\mathbb K),\quad{} \left\Vert AB\right\Vert \leq \left\Vert A\right\Vert \left\Vert B\right\Vert.$$
>
>- Subordinate norms are matrix norms. There are matrix norms that are not subordinate norms, for example the Frobenius norm:
>
>$$\left\Vert A\right\Vert_F = \sqrt{\mathrm{tr}(A^{*}A)}.$$
>
>- For any invertible matrix $A\in M_n(\mathbb{K})$, its condition number associated with a subordinate norm $\left\Vert \cdot \right\Vert$ is defined by
>
>$$\kappa(A) = \Vert A \Vert \Vert A^{-1} \Vert.$$
>
>- For $A\in M_n(\mathbb{K})$, the spectral radius of $A$ is defined by
>
>$$\rho(A)=\max\limits_{\lambda\in \text{Sp(A)}} \vert\lambda\vert.$$
>
>- If $A$ is a normal matrix ($A^*A = AA^*$), then $\left\Vert A \right\Vert_2 = \rho(A)$.

> **Performance remark before starting:** In this PC, we study iterative methods and their convergence. In practice, these methods are especially useful for sparse matrices, because they can fully use the sparse structure. In some implementation questions, we will pay particular attention to sparsity and computational performance; this will be stated explicitly. In the other questions, we may use less efficient code for pedagogical reasons, for example by storing all iterates or by not using the matrix structure in order to keep the code simple.

## Exercise 1: Spectral radius

### Question 1

Show that, for any subordinate norm, $\rho(A)\leq \left\Vert A \right\Vert$. Show that this property is also true for any matrix norm.

### Question 2

Show that for every $\varepsilon>0$ and every matrix $A\in M_n(\mathbb{C})$, there exists a subordinate norm such that $\left\Vert A \right\Vert \leq \rho(A)+\varepsilon$.

*Hint: triangularize $A$ in a suitable basis.*

### Question 3

Show that the following conditions are equivalent:

$1$. $\lim\limits_{n\rightarrow \infty} A^n=0$,

$2$. $\forall x\in \mathbb{C}^n,\ \lim\limits_{n\rightarrow \infty} A^nx=0$,

$3$. $\rho(A)<1$,

$4$. There exists a subordinate norm such that $\left\Vert A \right\Vert<1$.

*Hint: show that $1. \Rightarrow 2. \Rightarrow 3. \Rightarrow 4. \Rightarrow 1.$*

### Question 4

Let $B\in M_n(\mathbb{K})$ be such that $I-B$ is invertible, and let $c\in \mathbb{K}^n$. Consider an iterative method of the form

$$ \left\{ \begin{aligned} &x_0 \in \mathbb{K}^n\\ &x_{k+1} = Bx_k + c \quad \forall~k\in\mathbb{N}. \end{aligned}\right. $$

Show that the sequence $\left(x_k\right)$ converges for every $x_0\in\mathbb{K}^n$ if and only if $\rho(B)<1$. If $B$ is normal, show that the error in the $\left\Vert \cdot \right\Vert_2$ norm is bounded by a geometric sequence with ratio $\rho(B)$. What can be said about the error for an arbitrary matrix? *You may use Question 2.*

### Question 5

Show that, for any matrix norm,

$$\rho(A)=\lim\limits_{n\rightarrow \infty} \left\Vert A^n \right\Vert^{\frac{1}{n}}.$$

*Hint: consider the matrix $\frac{1}{\rho(A)+\varepsilon}A$.*

## Exercise 2: Stationary iterative methods

>Let $A\in M_n(\mathbb{R})$ be invertible and let $b\in\mathbb{R}^n$. Recall that, in general, to solve $Ax=b$ iteratively, we introduce a splitting
>
>$$ A=M-N $$
>
>of $A$, where $M$ is "easier to invert than $A$", and consider the sequence $x_k$ defined by
>
>$$\label{methode_it} x_{k+1} = M^{-1}\left(Nx_k + b\right). \qquad{} (1) $$
>
>The Jacobi method corresponds to taking $M$ equal to the diagonal of $A$, and the Gauss--Seidel method corresponds to taking $M$ equal to the lower triangular part of $A$.

### Question 1

1) Give a necessary and sufficient condition on $M^{-1}N$ for the iterative method (1) to converge to $x=A^{-1}b$ for every initial value $x_0\in\mathbb{R}^n$.

2) In how many iterations does the method converge if $M=A$ in (1)?

3) Let $A\in\mathbb{R}^{N\times N}$ be triangular. Show that the Jacobi and Gauss--Seidel algorithms converge exactly in at most $N$ iterations for this matrix.

> **Remark:** In the more general case where $A$ does not have these properties, it is very rare for these methods to converge in a finite number of iterations. In the rest of the exercise, we therefore study their convergence rate, which is related to the spectral radius of the iteration matrix.

### Question 2

>Let $B_J$ be the matrix $M^{-1}N$ associated with the Jacobi method, and let $B_{GS}$ be the matrix $M^{-1}N$ associated with the Gauss--Seidel method. The goal of this question is to prove that, if $A$ is tridiagonal,
>
>$$ \label{GSJ}\rho(B_{GS}) = \rho(B_J)^2. \qquad{} (2)$$

1) Let $A$ be the tridiagonal matrix

$$
A = \left( \begin{array}{cccccc}
    a_1    & b_1    & 0       &\dots   & 0 \\
    c_2    & a_2    & b_2     & \ddots& \vdots  \\
    0      & \ddots & \ddots &\ddots  &0\\
    \vdots & \ddots & \ddots &\ddots  & b_{N-1}\\
    0      & \dots  &  0   &   c_N  & a_N
  \end{array}\right).
$$

Show that, for every $\lambda\in\mathbb{C}^*$,

$$
\det \left( \begin{array}{cccccc}
    a_1   &  \lambda^{-1} b_1    & 0       &\dots   & 0 \\
    \lambda c_2    & a_2    &  \lambda^{-1} b_2     & \ddots& \vdots  \\
    0      & \ddots & \ddots &\ddots  &0\\
    \vdots & \ddots & \ddots &\ddots  &  \lambda^{-1} b_{N-1}\\
    0      & \dots  &  0   &   \lambda c_N  & a_N
  \end{array}\right) = 
  \det \left( \begin{array}{cccccc}
    a_1    & b_1    & 0       &\dots   & 0 \\
    c_2    & a_2    & b_2     & \ddots& \vdots  \\
    0      & \ddots & \ddots &\ddots  &0\\
    \vdots & \ddots & \ddots &\ddots  & b_{N-1}\\
    0      & \dots  &  0   &   c_N  & a_N
  \end{array}\right).
$$

2) Write $A=L+D+U$, where $L$ is the lower triangular part of $A$ without the diagonal, $D$ is the diagonal of $A$, and $U$ is the upper triangular part of $A$ without the diagonal. Show that $\lambda$ is an eigenvalue of $B_J$ if and only if

$$P_J(\lambda) := \det \left(\lambda D + L + U \right) = 0,$$

and that $\lambda$ is an eigenvalue of $B_{GS}$ if and only if

$$P_{GS}(\lambda) := \det \left(\lambda D + \lambda L + U \right) = 0.$$

3) Using Question 1), deduce that

$$ P_{GS} (\lambda^2) = \lambda^n P_J(\lambda) \qquad\forall~\lambda\in\mathbb{C},$$

and prove (2).

4) What can be deduced about the convergence of the Jacobi and Gauss--Seidel methods when $A$ is tridiagonal?

### Question 3

Consider

$$
A_1=\begin{pmatrix}
2 & -1 & 1 \\ 
2 & 2 & 2 \\
-1 & -1 & 2
\end{pmatrix}
\quad{} \text{and}\quad{}
A_2=\begin{pmatrix}
1 & 2 & -2 \\ 
1 & 1 & 1 \\
2 & 2 & 1
\end{pmatrix}.
$$

For each matrix, compute $\rho(B_{GS})$ and $\rho(B_J)$. What can be concluded about the convergence of the Jacobi and Gauss--Seidel methods in the general case?

In [ ]:
# Your code here


## Exercise 3: Fixed-step gradient method

>Let $A\in M_n(\mathbb{R})$ be symmetric positive definite and let $b\in\mathbb{R}^n$. To solve $Ax=b$ numerically, we look for the minimum of the function $f:\mathbb{R}^n\to \mathbb{R}$ defined by
>
>$$ f(x) = \frac{1}{2} \langle Ax,x\rangle - \langle b,x\rangle. $$
>
>We consider a fixed-step gradient method, that is, a sequence of the form
>
>$$ \left\{ \begin{aligned} &x_0 \in \mathbb{R}^n\\ &x_{k+1} = x_k - \alpha \nabla f(x_k) \quad{} \forall~k\in\mathbb{N}. \end{aligned}\right.  \quad{} (4)$$

### Question 1

Compute $\nabla f(x)$ and check that $f$ has a unique minimum, attained at $x^* = A^{-1}b$.

### Question 2: Convergence analysis

Let $0 < \lambda_1 \le \dots \le \lambda_N$ be the eigenvalues of $A$.

**<font color=black><span style="background-color:skyblue">To hand in:</span></font>**

1) Show that the sequence $(x_k)_{k\in\mathbb{N}}$ converges for every $x_0$ if and only if

$$0<\alpha<\frac{2}{\rho(A)}.$$

*Hint: use the results of Exercise 1.*

2) For every $\alpha \in ]0,\frac{2}{\rho(A)}[$, give and justify an error estimate of the form

$$\left\Vert x_k - x\right\Vert_2 \leq (g(\alpha))^k \left\Vert x_0 - x\right\Vert_2.$$

Give $g(\alpha)$ in terms of the eigenvalues of $A$. Then give the value of $\alpha_{opt}\in]0,\frac{2}{\rho(A)}[$ that minimizes $g(\alpha)$ (a detailed proof is not required, but an explanation is appreciated), and express $g(\alpha_{opt})$ in terms of $\kappa_2(A)$, the condition number of $A$ in the 2-norm. What does this estimate suggest about the convergence speed of the fixed-step gradient method when the condition number of $A$ is large?

*Hint: recall that $A$ is symmetric positive definite, so its 2-norm is its spectral radius, and its condition number in the 2-norm can be written in terms of the eigenvalues $\lambda_1$ and $\lambda_N$.*

**<font color=black><span style="background-color:skyblue">End of hand-in section</span></font>**

*The next question does not have to be handed in, but it can be used to check the results obtained in the previous question on a simple example.*

### Question 3: Implementation

>In practice, the algorithm described in (4) is used with two stopping criteria: a maximum number of iterations and a tolerance on the residual, for example $\|A x_k-b\| \le Tol$.

1) Implement a function that takes as input a matrix $A$, a vector $b$, a step size $\alpha$, an initial value $x_0$, a maximum number of iterations $N_{max}$, and a tolerance $Tol$. The function should return the iterates $x_k$ obtained with the fixed-step gradient method, together with the residual error $\Vert Ax_k-b\Vert$ at each iteration.

**WARNING:** In Python, the product of a matrix `A` and a vector `x` is not computed with `A*x`. Use `np.dot(A,x)` or `A.dot(x)` instead. The second option is also compatible with sparse matrices.

> **Performance remark:** Storing all iterates is useful only for studying the algorithm and uses unnecessary memory in practical computations.

In [ ]:
# Your code here


2) Test your algorithm on the following $2\times 2$ system:

$$
    A = \begin{pmatrix}
    1 & 0 \\
    0 & 7
    \end{pmatrix},\qquad{} b=\begin{pmatrix}
    2 \\ -1
    \end{pmatrix}.
$$

Use the initial value $x_0 = \begin{pmatrix} 1 \\1 \end{pmatrix}$.

Run the code below for different values of $\alpha$. Observe the behaviour of the iterates with respect to the level curves of the function

$$f:x\mapsto \frac{1}{2} \langle Ax,x\rangle - \langle b,x\rangle,$$

and study the convergence rate. Compare it with the theoretical estimate $g(\alpha)$ obtained in the previous question.

In [ ]:
# Function f used to plot its level curves
def f(A, b, x1, x2):
    """
    Compute f(x)=1/2*<Ax,x> - <b,x> for N=2 and x=(x1,x2).
    This implementation accepts arrays containing many values of x1 and x2
    (compatible with meshgrid).
    """
    return 1/2*((A[0,0]*x1+A[0,1]*x2)*x1 + (A[1,0]*x1+A[1,1]*x2)*x2) - b[0]*x1 - b[1]*x2


In [ ]:
# Grid used to plot the level curves of f
nx1    = 1000
nx2    = 1000
x1     = np.linspace(-0.5, 2.5, nx1)
x2     = np.linspace(-1.5, 1.5, nx2)
x1, x2 = np.meshgrid(x1, x2)

# Initialization
A     = np.array([[1,0],[0,7]])
b     = np.array([2,-1])
x1_th = 2
x2_th = -1/7
z     = f(A, b, x1, x2)
x0    = np.ones(len(b)) 


# Compute the iterates of the fixed-step gradient method
alpha = 0.1
tab_x_1, tab_err_1 = fixed_step_gradient(A, b, alpha, x0)

# Plot the level curves of f and the successive values of x_k
plt.figure(figsize=(16,5))
plt.contour(x1, x2, z)
plt.scatter(tab_x_1[0,:],tab_x_1[1,:], label=r"iterates $x_k$")
plt.title(r"Level curves of f, $\alpha$ = ??")
plt.legend()
plt.show()

# Study the convergence rate
tabk = np.arange(0,len(tab_err_1))
g = max(1-alpha,7*alpha-1)
plt.figure(figsize=(16,5))
plt.scatter(tabk , tab_err_1, label=r"$e_k=||Ax_k-b||_2$")
plt.plot(tabk , g**tabk, label=r"$(g(\alpha))^k$")
plt.yscale("log")
plt.xlabel("k")
plt.ylabel("Error")
plt.legend()
plt.show()


### Question 4: Back to the 1D Laplacian

>We return to the problem
>
>$$
\left\{ 
\begin{aligned}
&-u''(x)   = x^3        \quad{} x  \in ]0,1[ \\ 
& u(0) = 0 = u(1)
\end{aligned}
\right. \quad{} (6)
$$
>
>and to the associated discrete problem already studied in the previous PC:
>
>$$
AU =F \quad{} \text{where}\quad{}  A = 
    \begin{pmatrix}
      2/h^2    & -1/h^2    & 0       &\dots   & 0 \\
      -1/h^2    & 2/h^2    & -1/h^2     & \ddots& \vdots  \\
      0      & \ddots & \ddots &\ddots  &0\\
      \vdots & \ddots & \ddots &\ddots  & -1/h^2\\
      0      & \dots  &  0   &   -1/h^2  & 2/h^2
    \end{pmatrix},
    \quad{} b = 
    \renewcommand{\arraystretch}{1.2}
    \begin{pmatrix}
      x_1^3 \\ x_2^3 \\ \vdots \\ x_{N-1}^3 \\ x_N^3
    \end{pmatrix}, \quad{} h=\frac{1}{N+1}, \qquad{} (7)
$$
>
>where $x_i=ih$, $i=1,\ldots, N$, correspond to a discretization of the interval $[0,1]$, and the components $u_i$ of $U$ are expected to approximate the values $u(x_i)$ of the solution of (6).
>
> **Performance remark:** For the larger Laplacian matrix, we avoid unnecessary computations and storage and return to a more efficient implementation adapted to the sparse structure.

1) Reimplement the fixed-step gradient algorithm as before, but without storing all intermediate iterates. Make sure that the matrix-vector products are compatible with sparse storage: use *A.dot(x)* rather than *np.dot(A,x)*.

Now use the fixed-step gradient algorithm to approximate the solution of (7), with $N=63$, using the optimal step $\alpha$ obtained in Question 2 (you may use sparse-library functions for this), and with $x_0=(1,0,\dots,0)^T$, $N_{\max}=10^3$, and $Tol=10^{-5}$. Display the number of iterations performed and the residual error $\|Ax^k-b\|$ at the last iteration.

*Use the sparse structure of $A$, and therefore build $A$ with `scipy.sparse.diags`. To determine the optimal step, you may use the theoretical formulas for the eigenvalues of $A$ seen in class, or compute numerically the smallest and largest eigenvalues of $A$ with `scipy.sparse.linalg.eigs` (renamed `sparse_eigs` in the header).*

In [ ]:
# Your code here


In [ ]:
# Your code here


2) Study numerically the convergence rate obtained. Plot the error $\|Ax^k-b\|$ as a function of $k$ on a semilogarithmic scale. Compare it with the theoretical upper bound.

In [ ]:
# Your code here


3) Plot the discrete solution $u_i$ obtained after $10^3$ iterations, together with the exact solution. Comment.

In [ ]:
# Your code here


## Exercise 4: Conjugate gradient method (CG)

>Let $A$ be a real $n \times n$ symmetric positive definite matrix. Such a matrix defines the inner product
>
>$$ \langle x,y \rangle_A = x^TAy.$$
>
>Two vectors $d_1$ and $d_2$ are said to be conjugate with respect to $A$ if they are orthogonal for the inner product $\langle \cdot,\cdot \rangle_A$. In the following, the associated norm is denoted by $\|\cdot \|_A=\sqrt{\langle \cdot,\cdot \rangle_A}$.
>
>The conjugate gradient algorithm is described below:
>
>**Initialization:** Given a real symmetric positive definite matrix $A\in \Bbb{R}^{n\times n}$, $b\in \Bbb{R}^n$, $x_0 \in \mathbb{R}^n$, and a tolerance $Tol$:
>- Set $i=0$.
>- Compute the first residual $g_0 = (Ax_0-b)$ and the first descent direction $d_0 = -g_0$.
>- While $\Vert g_i\Vert > Tol$ and $i<n$:
>    - Compute the new iterate
>      $$ x_{i+1} = x_i-\gamma_i d_i \quad{}\text{with}\quad{} \gamma_i = \frac{d_i^Tg_i}{d_i^TAd_i} = \frac{\langle d_i,g_i\rangle}{\langle d_i,d_i\rangle_A}.$$
>    - Compute the new residual $g_{i+1} = Ax_{i+1}-b$.
>    - Compute the new descent direction
>      $$ d_{i+1} = -g_{i+1}+\beta_id_i \quad{}\text{with}\quad{} \displaystyle \beta_i = \frac{g_{i+1}^TAd_i}{d_i^TAd_i} = \frac{\langle g_{i+1},d_i\rangle_A}{\langle d_i,d_i\rangle_A}.$$
>
>One can show that this algorithm gives the following error estimate, which is better than the one obtained for the fixed-step gradient method:
>
>$$\left\Vert x_k - x\right\Vert_A \leq 2\left(\frac{\sqrt{\kappa_2(A)}-1}{\sqrt{\kappa_2(A)}+1}\right)^k \left\Vert x_0 - x\right\Vert_A.$$
>
>Moreover, one can also show (see Question 4 of this exercise) that CG converges in at most $n$ iterations. In practice, one almost never performs all $n$ iterations: the algorithm is stopped as soon as the residual error falls below a prescribed threshold.

### Question 1: Implementation

1) Show that, in the CG algorithm described above,

$$g_{i+1} = g_i - \gamma_i Ad_i.$$

What is the minimum number of matrix-vector products needed for one CG iteration? Which product(s) should be stored?

2) Implement the conjugate gradient algorithm. *Use as few matrix-vector products as possible.*

>**Performance remark:** At first, you may store all iterates $x^k$ in order to study the method.

3) Test your algorithm on the $2\times 2$ system from Exercise 3, Question 3, part 2). Plot again the level curves of the associated function $f$, together with the different iterates. Compare with the fixed-step gradient method and comment on the different behaviours.

In [ ]:
# Your code here


In [ ]:
# Your code here


### Question 2: Back to the 1D Laplacian

Reimplement the conjugate gradient algorithm as before, but without storing all intermediate iterates. Make sure that the matrix-vector products are compatible with sparse storage: use *A.dot(x)* rather than *np.dot(A,x)*.

Use this algorithm to compute an approximate solution of problem (7). Check that convergence is obtained in at most $N$ iterations, and plot the solution. For this problem, can the conjugate gradient algorithm be useful compared with the direct methods based on LU factorization studied in the previous PC?

In [ ]:
# Your code here


In [ ]:
# Your code here


### Question 3: Back to the 2D Laplacian

>We now consider the problem
>
>$$
\left\{ 
\begin{aligned}
&-\Delta u(x,y) = 1       \quad{} (x,y)  \in ]0,1[^2 \\ 
& u(0,y) = 0 = u(1,y) = u(x,0) = u(x,1)
\end{aligned}
\right. \quad{} (8)
$$
>
>and the associated discrete problem already studied in the previous PC:
>
>$$
AU =F \quad{} \text{where}\quad{}  A = 
\left( \begin{array}{c|c|c|c|c}
      D    & -I/h^2    &0 &\dots   & 0 \\ \hline
      -I/h^2    & D    & \ddots&  \ddots &\vdots  \\ \hline
      0    &  \ddots   & \ddots & \ddots& 0  \\ \hline
      \vdots   & \ddots &\ddots & \ddots &-I/h^2\\ \hline
      0      & \dots & 0 &   -I/h^2  & D
    \end{array}\right), \quad{}
    D = \left( \begin{array}{ccccc}
      4/h^2    & -1/h^2  & 0  & \dots   & 0 \\
      -1/h^2    & 4/h^2    & \ddots& \ddots& \vdots  \\   
      0      & \ddots & \ddots & \ddots  & 0\\
      \vdots & \ddots & \ddots  & \ddots & -1/h^2\\
      0      & \dots  & 0 & -1/h^2  & 4/h^2
    \end{array}\right) \in\mathbb{R}^{N},
    \qquad{} 
    \quad{} F = 
    \renewcommand{\arraystretch}{1.2}
    \begin{pmatrix}
      1 \\ 1 \\ \vdots \\ 1 \\ 1
    \end{pmatrix}, \quad{} h=\frac{1}{N+1}, \qquad{} (9)
$$
>
>where $(x_i,y_j)=(ih,jh)$, $i,j=1,\ldots, N$, correspond to a discretization of the square $[0,1]^2$, and the components of $U$ are expected to approximate the values $u(x_i,y_j)$ of the solution of (8).

1) Build the matrix of the discretized 2D Laplacian in sparse format. You may use *scipy.sparse.diags*.

In [ ]:
# Your code here


In [ ]:
# Test cell
N = 2
M = minus_laplacian_2D_sparse(N)
print(M)


2) Use the CG algorithm to compute an approximate solution of system (9). Plot the solution using the code below. For this problem, can the conjugate gradient algorithm be useful compared with the direct methods based on LU factorization studied in the previous PC?

In [ ]:
N = 63
A = minus_laplacian_2D_sparse(N)
b = np.ones(N**2)


# Grid used to plot the solution
nx     = N
ny     = N
dx     = 1./(nx+1)
dy     = 1./(ny+1)
x      = np.linspace(1,nx,nx)*dx
y      = np.linspace(1,ny,ny)*dy
xx, yy = np.meshgrid(x, y, indexing='xy')

# Solution
x0               = np.zeros(nx*ny) ; x0[0] = 1.
U_GC, tab_err_GC = conjugate_gradient_sparse(A, b, x0, it_max = 10**3, tol = 10**(-5))
Uij              = np.reshape(U_GC, (nx,ny))


# Plot
plt.figure()
plt.contourf(xx,yy,np.reshape(U_GC,(nx,ny)))
plt.colorbar()
plt.show()


### Question 4: Theoretical analysis

In this question, we study the properties of the conjugate gradient algorithm described at the beginning of the exercise.

1) Let $x^*$ be the solution of $Ax=b$. Show that $\gamma_i$ minimizes

$$\gamma \mapsto \|x_i-\gamma d_i -x^*\|_A^2.$$

2) Show that $\beta_i$ is defined so that

$$\langle d_{i+1},d_i\rangle_A = 0,$$

i.e. so that each direction $d_i$ is $A$-orthogonal to the next direction $d_{i+1}$.

3) Show the following identities:

$$\langle g_{i+1}, d_i\rangle = 0, \qquad
  \langle d_i, d_i\rangle_A = -\langle d_i, g_i\rangle_A.$$

Deduce that

$$\langle g_{i+1},g_i\rangle = 0,$$

i.e. that the gradient $g_i$ is orthogonal to the next gradient $g_{i+1}$.

4) Show that

$$\langle g_{i+1},g_j\rangle=0
\quad{} \text{and} \quad{}
\langle d_{i+1},d_j\rangle_A = 0
\quad \forall j = 0,\ldots,i.$$

Conclude that the conjugate gradient algorithm converges in at most $n$ iterations.

5) Assume that

$$\left\Vert x_k - x\right\Vert_A = \min_{\substack{P \in \mathbb{R}_k[X] \\ P(0)=1}} \left\Vert P(A)(x_0 - x)\right\Vert_A.$$

Deduce that, if the minimal polynomial of $A$ has degree $m$, then the conjugate gradient algorithm converges in at most $m$ iterations.